In [15]:
import os
import time
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
## Utility methods

def build_user_message(message):
    return [{"role": "user", "content": message}]

## Gemini Example

In [16]:
gemini_base_url = os.getenv("GEMINI_BASE_URL")
google_api_key = os.getenv("GOOGLE_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL")
gemini_model_validator = os.getenv("GEMINI_MODEL_VALIDATOR")

gemini = OpenAI(base_url=gemini_base_url, api_key=google_api_key)

In [10]:
## Example 1:

start = time.time()
response = gemini.chat.completions.create(model=gemini_model, messages=[{"role":"user", "content": "what is 2+2?"}])

print(response.choices[0])

print(response.choices[0].message.content)
print(time.time() - start)

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='2 + 2 = 4', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, extra_content={'google': {'thought_signature': 'EnMKcQFpFH0Tuff2eIP3FkT0Fn+FSvABujI6tmmpjXC9J8hOOzVuS4FL0ABKPjQ5PJudCklBde8opMStC4YohbGNGETkX6GdNoF68GJ9au516cTnPere+w90ZPEO23IAixtNUphFZB/TyQfu7PqbFwqbCFco'}}))
2 + 2 = 4
2.584904432296753


In [11]:
## Example 2:

start = time.time()
response = gemini.chat.completions.create(model=gemini_model, messages=[{"role":"user", "content":"Tell me two fun facts!"}])

print(response.choices[0].message.content)
print(time.time() - start)

Here are two fun facts for you:

1. **Sea otters hold hands when they sleep.** They do this to keep from drifting apart in the current, creating what is known as a "raft." Sometimes, they will even wrap themselves in long strands of kelp to act as an anchor while they snooze.

2. **Bananas are berries, but strawberries aren’t.** In botanical terms, a berry must have seeds on the inside and develop from a single flower with one ovary. By that definition, bananas (along with watermelons and pumpkins) qualify as berries, while strawberries—which have their "seeds" (achenes) on the outside—do not!
5.372981786727905


## Ollama Example

In [12]:
ollama_base_url = os.getenv("OLLAMA_BASE_URL")
ollama_api_key = os.getenv("OLLAMA_API_KEY")
ollama_model = os.getenv("OLLAMA_MODEL")

ollama = OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)

In [13]:
response = ollama.chat.completions.create(model=ollama_model, messages=[{"role": "user", "content": "What is 2 + 2?"}])

print(response.choices[0].message.content)

2 + 2 equals 4.


## GEMINI Example - questions and answers based requests with validations using another model

In [33]:
message = build_user_message("Please propose a hard, challenging question to assess someone's IQ. Respond only with the question.")
response = gemini.chat.completions.create(model=gemini_model, messages=message)
question = response.choices[0].message.content
print("Question: {}".format(question))

Question: A sequence of numbers follows a hidden logical and mathematical rule: 2, 6, 12, 20, 30, 42, 56, 72, ... What is the exact value of the 100th term in this sequence, and what is the algebraic formula that generates the $n$-th term?


In [34]:
message = build_user_message(question)
response = gemini.chat.completions.create(model=gemini_model_validator, messages=message)
answer = response.choices[0].message.content

In [35]:
from IPython.display import Markdown, display

display(Markdown(answer))

### **1. Algebraic Formula for the $n$-th Term**

The general formula to generate the $n$-th term ($a_n$) of this sequence is:

$$a_n = n(n + 1) \quad \text{or} \quad a_n = n^2 + n$$

---

### **2. Value of the 100th Term**

Using the formula for $n = 100$:

$$a_{100} = 100 \times (100 + 1)$$
$$a_{100} = 100 \times 101 = \mathbf{10,100}$$

---

### **Explanation of the Pattern**

There are a few simple ways to see how this rule works:

1. **Product of Consecutive Integers:**
   * $1^{\text{st}}$ term ($n=1$): $1 \times 2 = 2$
   * $2^{\text{nd}}$ term ($n=2$): $2 \times 3 = 6$
   * $3^{\text{rd}}$ term ($n=3$): $3 \times 4 = 12$
   * $4^{\text{th}}$ term ($n=4$): $4 \times 5 = 20$
   * $\dots$
   * $n^{\text{th}}$ term: $n \times (n + 1)$

2. **Square plus $n$:**
   * $1^2 + 1 = 2$
   * $2^2 + 2 = 6$
   * $3^2 + 3 = 12$
   * $4^2 + 4 = 20$
   * $\dots$
   * $n^2 + n$

3. **Differences Between Terms:**
   The differences between consecutive terms increase by $2$ each time ($4, 6, 8, 10, 12, 14, 16 \dots$). A constant second difference confirms that the sequence is quadratic in the form of $a_n = n^2 + n$.

In [37]:
message = f"""
Here is a question:
{question}

And here is a possible answer that might be correct or incorrect:
{answer}

Please evaluate if the answer is correct or incorrect.
"""

print(message)
response = gemini.chat.completions.create(model=gemini_model, messages=build_user_message(message))
print(response.choices[0].message.content)


Here is a question:
A sequence of numbers follows a hidden logical and mathematical rule: 2, 6, 12, 20, 30, 42, 56, 72, ... What is the exact value of the 100th term in this sequence, and what is the algebraic formula that generates the $n$-th term?

And here is a possible answer that might be correct or incorrect:
### **1. Algebraic Formula for the $n$-th Term**

The general formula to generate the $n$-th term ($a_n$) of this sequence is:

$$a_n = n(n + 1) \quad \text{or} \quad a_n = n^2 + n$$

---

### **2. Value of the 100th Term**

Using the formula for $n = 100$:

$$a_{100} = 100 \times (100 + 1)$$
$$a_{100} = 100 \times 101 = \mathbf{10,100}$$

---

### **Explanation of the Pattern**

There are a few simple ways to see how this rule works:

1. **Product of Consecutive Integers:**
   * $1^{\text{st}}$ term ($n=1$): $1 \times 2 = 2$
   * $2^{\text{nd}}$ term ($n=2$): $2 \times 3 = 6$
   * $3^{\text{rd}}$ term ($n=3$): $3 \times 4 = 12$
   * $4^{\text{th}}$ term ($n=4$): $4 \times 

In [38]:
display(Markdown(print(response.choices[0].message.content)))

The provided answer is **completely correct**. 

Here is a brief evaluation of why:

1. **Algebraic Formula:** The formula $a_n = n(n+1)$ (or expanded as $n^2 + n$) accurately generates every term in the sequence when tested for $n = 1, 2, 3, \dots$
2. **100th Term:** Substituting $n = 100$ into the formula yields $100 \times 101 = 10,100$, which is mathematically accurate.
3. **Explanation:** The logic provided (product of consecutive integers, square plus $n$, and the method of finite differences) correctly proves both the formula and the pattern.


<IPython.core.display.Markdown object>

# Excercise | Pick any business use case and follow the instructions

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>
        </td>
    </tr>
</table>

In [44]:
## Pick a business

message = build_user_message("Pick a business area that might be worth exploring for an Agentic AI opportunity. Prefererably related with personal time and calendar management for complete coverage of personal responsibilities")
response = gemini.chat.completions.create(model=gemini_model, messages=message)
business_use_case = response.choices[0].message.content
print(business_use_case)

An exceptionally high-potential business area for Agentic AI is **"Holistic Life Operations & Family Logistics Management"** (often referred to as the **Chief of Staff for Personal Life**). 

While current tools (Google Calendar, Todoist, Calendly) manage *fragments* of time and tasks, they require heavy human intervention to coordinate. An Agentic AI system wouldn't just schedule; it would **anticipate, negotiate, execute, and adapt** across all domains of personal and family responsibility.

Here is a breakdown of why this is worth exploring, what the agentic opportunity looks like, and how it achieves "complete coverage."

---

### The Vision: The Personal Life "Chief of Staff"
Unlike a chatbot that waits for a prompt, an Agentic AI acts autonomously with a defined goal, using tools, browsing the web, and interacting with other humans and software. 

In personal time and calendar management, the ultimate product is an agent that manages the friction between **Time, Tasks, Relationsh

In [45]:
message = build_user_message("Business context: \n{}".format(business_use_case) + "\n. Query: present a pain-point in this usecase - something challenging that might be ripe for an Agentic solution")
response = gemini.chat.completions.create(model=gemini_model_validator, messages=message)
pain_point = response.choices[0].message.content

print(pain_point)

A prime, highly challenging pain point in this space that is exceptionally ripe for an Agentic AI solution is **"The Domino Effect Chaos" (Dynamic Exception Handling & Multi-Party Micro-Negotiation).**

---

### The Pain Point: The "Domino Effect" of Real-Life Disruptions

#### The Human Problem
In personal and family life, schedules are not static; they are highly interconnected, fragile webs. When **one single variable breaks**, it triggers a catastrophic domino effect of 5–10 secondary tasks, schedule shifts, and human negotiations.

**Example Scenario:**
At 1:15 PM, a parent gets a text from the school nurse: *"Your child has a 101°F fever. Please pick them up within 45 minutes."*

**The resulting friction (What the parent must do manually right now):**
1.  **Evaluate Priorities:** Check both parents' work calendars to see who has the "least unmovable" meeting between 1:30 PM and 3:00 PM.
2.  **Negotiate:** Rapidly text/call the spouse to agree on who leaves work.
3.  **Reschedule 

In [46]:
message = build_user_message("With business use case: \n {}".format(business_use_case) + "\n\n" + "and pain point: \n {}".format(pain_point) + "\n\n" + "Qury: Propose the Agentic AI solution for this use case")
response = gemini.chat.completions.create(model=gemini_model_validator, messages=message)

ai_solution = response.choices[0].message.content

print(ai_solution)

Here is a blueprint and system architecture for an Agentic AI product designed to solve **Holistic Life Operations** and master the **Domino Effect Chaos**.

---

# Product Concept: **Aegis OS**
*The Autonomous Chief of Staff for Personal & Family Life*

---

## 1. System Architecture: The Multi-Agent Network
Rather than relying on a single LLM prompt, **Aegis OS** uses a **Hierarchical Multi-Agent Architecture (HMAA)**. Personal life is too complex for one prompt; it requires dedicated domain agents coordinated by a master reasoning engine.

```
                    ┌─────────────────────────┐
                    │   User & Inputs (Voice, │
                    │   SMS, Webhooks, Apps)  │
                    └────────────┬────────────┘
                                 │
                                 ▼
                    ┌─────────────────────────┐
                    │  Master Orchestrator    │
                    │   (Chief of Staff AI)   │
                    └────────────┬───────